# cube fitting example


In [ ]:
import sys
from pathlib import Path

import a.pyplot as plt
import numpy as np
from astropy import constants as const
from astropy import units as u
from astropy.io import fits
from spectral_cube import SpectralCube

from simplefit import fit_cube, fit_spectrum, plot_fit

In [ ]:
INPUT_FILE = "./example_data/ngc1512_meerkat_hi21cm_pbcorr_zoom_15arcsec_k.fits"
INPUT_MASK_FILE = "./example_data/ngc1512_meerkat_hi21cm_pbcorr_zoom_15arcsec_k_broad_mom0.fits"

In [ ]:
cube = SpectralCube.read(INPUT_FILE)
cube = cube.with_spectral_unit(u.km / u.s, velocity_convention="radio")

In [ ]:
mask = fits.getdata(INPUT_MASK_FILE).astype(bool)
plt.imshow(mask, origin="lower")

In [ ]:
cube_fit = fit_cube(cube, n_jobs=10, progress=True, mask=mask, ssa_size=10)

In [ ]:
print(f"Fitted pixels: {mask.sum()}")
print(f"Successful fits: {cube_fit.success[mask].sum()}")
print(f"Total fitted components: {len(cube_fit.component_table)}")
print(f"Maximum components in one pixel: {cube_fit.n_components.max()}")

In [ ]:
component_table = cube_fit.component_table
component_table = component_table[component_table["amplitude"] > 0.0]


plt.hist(cube_fit.component_table["fwhm"]/2.35, bins=100)
plt.show()

## Cube Fit Summary Maps


In [ ]:
first_center = cube_fit.component_map("center", component=1)
first_amplitude = cube_fit.component_map("amplitude", component=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
images = [
    axes[0].imshow(cube_fit.n_components, origin="lower", cmap="viridis"),
    axes[1].imshow(cube_fit.success, origin="lower", cmap="gray_r", vmin=0, vmax=1),
    axes[2].imshow(first_center, origin="lower", cmap="magma"),
]
axes[0].set_title("Number of components")
axes[1].set_title("Fit success")
axes[2].set_title("Component 1 center")
for ax, image in zip(axes, images):
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    fig.colorbar(image, ax=ax, shrink=0.85)
plt.show()


In [ ]:
component_table = cube_fit.component_table
valid_components = component_table[component_table['success'] & (component_table['fwhm'] > 0)]
min_fwhm = np.full(cube_fit.n_components.shape, np.nan, dtype=float)
for row in valid_components:
    y = int(row['y'])
    x = int(row['x'])
    value = float(row['fwhm'])
    if np.isnan(min_fwhm[y, x]):
        min_fwhm[y, x] = value
    else:
        min_fwhm[y, x] = min(min_fwhm[y, x], value)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
image = ax.imshow(min_fwhm, origin='lower', cmap='magma')
ax.set_title('Minimum FWHM per pixel')
ax.set_xlabel('x pixel')
ax.set_ylabel('y pixel')
fig.colorbar(image, ax=ax, shrink=0.85, label='FWHM')
plt.show()


## Save Sparse Component Table


In [ ]:
OUTPUT_TABLE = Path("./example_outputs/cube_fit_components_bigger.csv")
cube_fit.write_table(OUTPUT_TABLE)
OUTPUT_TABLE

## Interactive 3D Component Plot


In [ ]:
import numpy as np
import plotly.express as px

cube_fit.component_table["log(amplitude)"] = np.log10(cube_fit.component_table["amplitude"])
plot_components = cube_fit.component_table
plot_components = plot_components[
    plot_components["success"]
    & (plot_components["amplitude"] > 0.0)
    & np.isfinite(plot_components["center"])
    & np.isfinite(plot_components["fwhm"]) 
    & (plot_components["fwhm"] < 100.0)
]

plot_df = plot_components.to_pandas()

fig = px.scatter_3d(
    plot_df,
    x="x",
    y="y",
    z="center",
    color="log(amplitude)",      # point colour = intensity
    size="fwhm",            # point size = FWHM
    color_continuous_scale="magma",
    range_color=[0, 2],
    size_max=12,
    title="Fitted Gaussian Components in Position-Position-Velocity Space",
)

fig.update_traces(
    marker={
        "opacity": 0.2,
        "line": {"width": 0},
    }
)

fig.update_layout(
    scene={
        "xaxis_title": "x pixel",
        "yaxis_title": "y pixel",
        "zaxis_title": "center velocity",
        "yaxis": {"autorange": "reversed"},
    },
    margin={"l": 0, "r": 0, "b": 0, "t": 45},
)

fig.show()
fig.write_html("./example_outputs/cube_fit_3d.html", auto_open=True)